# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.


### 🧲 Quick Link to Codes

- [1. Setup Environment & Document Prep](#1-environment-setup--document-prep) 
- [2. Sending documents in batches to OpenAI for tasks](#2-sending-documents-in-batches-to-openai-for-tasks)
- [3. Evaluation using DeepEval](#3-evaluation-using-deepeval)
- [4. Enhancement Prompt](#4-enhancement-prompt)
- [5. Evaluation the Enhancement Prompt](#5-evaluation-the-enhancement-prompt)
- [6. Final Thoughts](#final-thoughts)

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

### 👉 Note 

- join the text to together and using split_text(). The reason of not using `split_documents()` is because for this type of work (summarization), the chance of using the metadata is low.   


- `split_text()` vs. `split_documents()`:   
<br>  

| Function            | Input Type       | Output           | Pros                                                                                        | Cons                                                               | Apply When                                                                                          |
| ------------------- | ---------------- | ---------------- | ------------------------------------------------------------------------------------------- | ------------------------------------------------------------------ | --------------------------------------------------------------------------------------------------- |
| `split_text()`      | `string`         | `list[str]`      | ✅ Simple<br>✅ Lightweight<br>✅ Good for summarization<br>✅ No metadata overhead             | ❌ No source tracking<br>❌ No page reference<br>❌ Not ideal for RAG | ✔ Full-document summarization<br>✔ One-off processing<br>✔ Small projects<br>✔ No need for citation |
| `split_documents()` | `list[Document]` | `list[Document]` | ✅ Preserves metadata<br>✅ Enables citation<br>✅ Better for RAG<br>✅ Scalable for production | ❌ Slightly more complex<br>❌ Requires structured input             | ✔ RAG systems<br>✔ QA with page reference<br>✔ Multi-document pipelines<br>✔ Production workflows   |



### 1. Environment Setup & Document Prep

In [7]:
%load_ext dotenv
%dotenv ../05_src/.secrets

import os 

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [ ]:
##### >>>  1. Import and split document  <<< ##### 

from langchain_community.document_loaders import PyPDFLoader

file_path = "../05_src/documents/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(f"Total Page:{len(docs)}")
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)  # pritn first page 
# print(f"{docs[2].page_content[:1000]}\n") print content from Page 3.  


Total Page:26
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025

{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-07-13T21:18:19-07:00', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_siteid': '72f988bf-86f1-41af-91ab-2d7cd011db47', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_method': 'Privileged', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_enabled': 'True', 'author': 'Aditya Challapally', 'moddate': '2025-07-13T21:18:19-07:00', 'source': '../05_src/documents/ai_report_2025.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}


In [ ]:
## join every thing into a single strings
document_text = ""
for page in docs: 
    document_text += page.page_content + "\n"   ## this will 
# display(document_text)  
print(f"Total Length: {len(document_text)} characters")

'pg. 1 \n \n \nThe GenAI Divide  \nSTATE OF AI IN \nBUSINESS 2025 \n \n \n \n \n \n \nMIT NANDA \nAditya Challapally \nChris Pease \nRamesh Raskar \nPradyumna Chari \nJuly 2025\npg. 2 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nNOTES \nPreliminary Findings from AI Implementation Research from Project NANDA \nReviewers: Pradyumna Chari, Project NANDA \nResearch Period: January – June 2025 \nMethodology: This report is based on a multi-method research design that includes \na systematic review of over 300 publicly disclosed AI initiatives, structured \ninterviews with representatives from 52 organizations, and survey responses from \n153 senior leaders collected across four major industry conferences. \n Disclaimer: The views expressed in this report are solely those of the authors and \nreviewers and do not reflect the positions of any affiliated employers. \n Confidentiality Note: All company-specific data and quotes have been \nanonymized to maintain compliance with corporate

Total Length: 53851 characters


In [10]:
from langchain_text_splitters  import RecursiveCharacterTextSplitter
import textwrap # help wrap text for single line output

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 4000, 
    chunk_overlap=500, 
    length_function = len, 
    add_start_index = True
)

chunks = text_splitter.split_text(document_text)

print(textwrap.fill(chunks[0],width=120))
print("")
print(f'Output: Split {len(document_text)} characters into {len(chunks)} chunks.' )

pg. 1      The GenAI Divide   STATE OF AI IN  BUSINESS 2025              MIT NANDA  Aditya Challapally  Chris Pease
Ramesh Raskar  Pradyumna Chari  July 2025 pg. 2                                    NOTES  Preliminary Findings from AI
Implementation Research from Project NANDA  Reviewers: Pradyumna Chari, Project NANDA  Research Period: January – June
2025  Methodology: This report is based on a multi-method research design that includes  a systematic review of over 300
publicly disclosed AI initiatives, structured  interviews with representatives from 52 organizations, and survey
responses from  153 senior leaders collected across four major industry conferences.   Disclaimer: The views expressed
in this report are solely those of the authors and  reviewers and do not reflect the positions of any affiliated
employers.   Confidentiality Note: All company-specific data and quotes have been  anonymized to maintain compliance
with corporate disclosure policies and  confidentiality agreeme

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


### 2. Sending documents in batches to OpenAI for tasks

[Back to Quick Link to Code Boxes 🧲](#-quick-link-to-codes)

In [ ]:
##### >>>  2. Sedning documents in batches to OpenAI  <<< ##### 

### Step 1: Connect to OpenAI SDK
from tqdm import tqdm
from openai import OpenAI
# import os

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})


# confirm chunks exist
print(f"Total chunks to process: {len(chunks)}")
chunks[1]

Total chunks to process: 16


"pg. 4 \n \nsystems that integrate with existing processes and improve over time. Vendors meeting \nthese expectations are securing multi-million-dollar deployments within months. \nWhile most implementations don't drive headcount reduction, organizations that have \ncrossed the GenAI Divide are beginning to see selective workforce impacts in customer \nsupport, software engineering, and administrative functions. In addition, the highest-\nperforming organizations report measurable savings from reduced BPO spending and \nexternal agency use, particularly in back-office operations. Others cite improved customer \nretention and sales conversion through automated outreach and intelligent follow-up \nsystems. These early results suggest that learning-capable systems, when targeted at \nspecific processes, can deliver real value, even without major organizational restructuring. \n \n3 THE WRONG SIDE OF THE GENAI DIVIDE: HIGH ADOPTION, \nLOW TRANSFORMATION \nTakeaway: Most organizations fall

In [ ]:
### Step 2: Submit chunks to OpenAI for tasks 

from IPython.display import display, Markdown, JSON

# Store partial summaries
partial_summaries = []

instructions = "You are a female humorous politician who speaks and writes in a soft, non-assertive, playful, and sarcastic tone." 

# helper function for summarize chunks
def summarize_chunk(chuck_text):
    """
    Summarize a chunk and return a validated JSON format.
    """

    base_prompt = f"""
    Create a summary that is concise, structured, and no longer than 1000 token. The final output needs to a valid JSON format so we can parse into a pydantic basemodel later. All values must be a single **strings** except tokens.
    
    JSON file key:
     - Organization : "MIT NANDA" - # single string
     - Author : "Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari" - # single string
     - Title : "The GenAI Divide STATE OF AI IN BUSINESS 2025" 
     - Relevance : "..."  # single string; do not output a nested object
     - Summary : "..."  # single string; do not output a nested object
     - Tone : "..." # single string; do not output a nested object
     - InputTokens:  # numbers 
     - OutputTokens  # numbers 

     Summarize this chuck:
     {chuck_text}
     """
    
    response = client.responses.create(
        model="gpt-4o-mini",
        instructions=instructions,
        input=base_prompt,
        temperature=0.5  # default temperature = 1
    )
    return response.output_text

## Loop to process all chuncks 
for i, chunk in enumerate(tqdm(chunks, desc="Summarizing chunks")):
    try:
        summary = summarize_chunk(chunk)
        partial_summaries.append(summary)
        print(f"✅ Chunk {i+1}/{len(chunks)} summarized")

    except Exception as e:
        print(f"❌ Failed at chunk {i+1}: {e}")


Summarizing chunks:   6%|▋         | 1/16 [00:04<01:00,  4.06s/it]

✅ Chunk 1/16 summarized


Summarizing chunks:  12%|█▎        | 2/16 [00:07<00:52,  3.75s/it]

✅ Chunk 2/16 summarized


Summarizing chunks:  19%|█▉        | 3/16 [00:11<00:51,  3.93s/it]

...

✅ Chunk 14/16 summarized


Summarizing chunks:  94%|█████████▍| 15/16 [01:02<00:04,  4.13s/it]

✅ Chunk 15/16 summarized


Summarizing chunks: 100%|██████████| 16/16 [01:08<00:00,  4.26s/it]

✅ Chunk 16/16 summarized


In [16]:
partial_summaries


['```json\n{\n  "Organization": "MIT NANDA",\n  "Author": "Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",\n  "Title": "The GenAI Divide STATE OF AI IN BUSINESS 2025",\n  "Relevance": "This report highlights the stark contrast in outcomes from GenAI investments, emphasizing the need for effective implementation strategies.",\n  "Summary": "Despite significant investment in GenAI, 95% of organizations see no return, with only 5% of AI pilots yielding substantial value. The divide stems from approach rather than model quality, with tools enhancing productivity but failing to impact P&L. Key barriers include limited disruption, an enterprise paradox, investment biases, and inadequate learning. Successful buyers demand customization and focus on business outcomes, while vendors that adapt quickly secure lucrative deployments.",\n  "Tone": "Playful and sarcastic, yet informative",\n  "InputTokens": 1000,\n  "OutputTokens": 800\n}\n```',
 '```json\n{\n  "Organization": "MIT

In [17]:
### Step 3: Parse into a pydantic basemodel

## 1. create a class 
from pydantic import BaseModel

class Summary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int


## 2. Use TypeAdapter to parse each chunk
from pydantic import TypeAdapter
import json

summary_adapter = TypeAdapter(Summary)
partial_summaries_objects = []

for raw_summary in partial_summaries:
    try:
        # Remove markdown wrapping ```json ... ```
        json_str = raw_summary.strip().strip("```json").strip("```").strip()
        summary_dict = json.loads(json_str)
        
        # Parse into Pydantic Summary object
        summary_obj = summary_adapter.validate_python(summary_dict)
        partial_summaries_objects.append(summary_obj)
    except Exception as e:
        print(f"Failed to parse chunk: {e}")

In [18]:
partial_summaries_objects

[Summary(Author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide STATE OF AI IN BUSINESS 2025', Relevance='This report highlights the stark contrast in outcomes from GenAI investments, emphasizing the need for effective implementation strategies.', Summary='Despite significant investment in GenAI, 95% of organizations see no return, with only 5% of AI pilots yielding substantial value. The divide stems from approach rather than model quality, with tools enhancing productivity but failing to impact P&L. Key barriers include limited disruption, an enterprise paradox, investment biases, and inadequate learning. Successful buyers demand customization and focus on business outcomes, while vendors that adapt quickly secure lucrative deployments.', Tone='Playful and sarcastic, yet informative', InputTokens=1000, OutputTokens=800),
 Summary(Author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide STATE OF AI IN 

In [ ]:
### Step 4: Summary Prompt 

# Combined Text
combined_text = "\n\n".join([s.Summary for s in partial_summaries_objects])

combined_text

prompt= f"""
Combine the following chunk summaries into a single summary that is concise, structured, and no longer than 1000 token. The final output needs to a valid JSON format so we can parse into a pydantic basemodel later. All values must be as a **strings** except tokens.

JSON file key:
- Organization : "MIT NANDA" - # single string
- Author : "Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari"
- Title : "The GenAI Divide STATE OF AI IN BUSINESS 2025" 
- Relevance : "..."  # single string; do not output a nested object
- Summary : "..."  # single string; do not output a nested object
- Tone : "..." # single string; do not output a nested object
- InputTokens:  # numbers 
- OutputTokens  # numbers 

Summarize this chuck:
{combined_text}
"""


response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=prompt,
    temperature=0.5  #reduce temperature=0.5
)


In [20]:
wrapped_text=combined_text
print(textwrap.fill(wrapped_text,width=120))

Despite significant investment in GenAI, 95% of organizations see no return, with only 5% of AI pilots yielding
substantial value. The divide stems from approach rather than model quality, with tools enhancing productivity but
failing to impact P&L. Key barriers include limited disruption, an enterprise paradox, investment biases, and inadequate
learning. Successful buyers demand customization and focus on business outcomes, while vendors that adapt quickly secure
lucrative deployments.  While many organizations are adopting GenAI tools, actual transformation remains elusive. Most
sectors show minimal structural changes despite significant investments. Only the Tech and Media industries exhibit
signs of disruption, while others lag behind. A composite AI Market Disruption Index reveals that despite some progress,
many industries are still struggling to integrate AI effectively into their workflows.  Despite extensive pilot activity
in seven out of nine sectors, organizations struggle t

In [21]:
final_summary_text = response.output_text
print(textwrap.fill(final_summary_text,width=120))

```json {   "Organization": "MIT NANDA",   "Author": "Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
"Title": "The GenAI Divide STATE OF AI IN BUSINESS 2025",   "Relevance": "The report highlights the challenges
organizations face in effectively implementing GenAI tools, revealing a significant gap between investment and actual
returns.",   "Summary": "Despite heavy investment in GenAI, 95% of organizations see no return, with only 5% of pilots
yielding value. The divide is attributed to approach rather than model quality, with barriers like limited disruption
and investment biases. While Tech and Media show signs of transformation, most sectors lag behind. Enterprises struggle
to transition from pilot projects to scalable solutions, often misallocating resources toward sales and marketing
instead of back-office automation. Users prefer adaptable AI tools like ChatGPT, while the emergence of 'Agentic AI'
aims to bridge the GenAI Divide by enhancing memory and learnin

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

### 3. Evaluation using DeepEval

[Back to Quick Link to Codes 🧲](#-quick-link-to-codes)

In [84]:
##### >>> Evaluation with DeepEval<<< #####

## Load all the required library at once
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric, SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel


## Defiend INPUT, OUTPUT for the metrices evaluation
INPUT=document_text
OUTPUT=final_summary_text


# Initialize empty dictionary at the top
evaluation_results = {}

In [ ]:
# ##### >>> 3.1 Evaluation: Answer Relevancy Metric (baseline)

# ## Link to Answer Revelancy: https://deepeval.com/docs/metrics-answer-relevancy

# model = GPTModel(
#     model="gpt-4o-mini",
#     temperature=0, # temperatre = 0 
#     default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
#     base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
# )

# metric = AnswerRelevancyMetric(
#     threshold=0.7,
#     include_reason=True,
#     model=model,
    
# )

# test_case = LLMTestCase(
#     input=INPUT,
#     actual_output=OUTPUT,
    
# )

# ## run the test
# metric.measure(test_case)


# ## output relevance metric 
# # print(metric.score)
# # print(metric.reason)
# display(Markdown(f'**Score**: {metric.score}'))
# display(Markdown(f'**Reason**: {metric.reason}'))

# evalu_relevance = {
#     "RelevanceScore": metric.score,
#     "RelevancenReason": metric.reason
# }

# with open("evaluation.json", "w") as f:
#     json.dump(evalu_relevance, f, indent=2)

Output()

1.0

In [ ]:
##### >>> 3.2 Evaluation: Summarization Metric

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,  
    # temperature=0 → deterministic. The model will almost always pick the “most likely” output. Best for: structured outputs, JSON, factual summaries.
    # temperature=1 → more creative / diverse. Best for: storytelling, playful prose,
    
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

metric = SummarizationMetric(
    threshold=0.5,  # default 0.5
    include_reason=True,
    model=model,
    verbose_mode=True,
    # truths_extraction_limit=20 , # default NONE
    assessment_questions = [
    "Does the output in a JSON format?",
    "Is the Author correct?",
    "Is the Title correct",
    "Is the Summary present",
    "Is the tone playful",
    "Is the tone soft and non-assertive?",
    "Is the tone sarcastic?",
    "Is the summary under 1000 tokens?"
    ],
    n = 8,
)


test_case = LLMTestCase(
    # input= document_text,
    input=INPUT,
    actual_output=OUTPUT,
    
)

## run the test
metric.measure(test_case)

## output summerization metric 

# print(metric.score)
# print(metric.reason)
display(Markdown(f'**Score**: {metric.score}'))
display(Markdown(f'**Reason**: {metric.reason}'))

# output to result dictioanry
evaluation_results["SummarizationScore"] = metric.score
evaluation_results["SummarizationReason"] = metric.reason



Output()

**************************************************

Summarization Verbose Logs

**************************************************

Truths (limit=None):
[
    "The report is titled 'The GenAI Divide: State of AI in Business 2025'.",
    "The report was produced by MIT NANDA and authored by Aditya Challapally, Chris Pease, Ramesh Raskar, and 
Pradyumna Chari.",
    "The research period for the report was from January to June 2025.",
    "The report is based on a multi-method research design that includes a systematic review of over 300 publicly 
disclosed AI initiatives.",
    "The report includes structured interviews with representatives from 52 organizations.",
    "The report includes survey responses from 153 senior leaders collected across four major industry 
conferences.",
    "The views expressed in the report are solely those of the authors and reviewers.",
    "All company-specific data and quotes in the report have been anonymized.",
    "The report states that 95% of organizations are getting zero return on their investment in GenAI.",
    "Only 5% of integrated AI pilots are extracting millions in value.",
    "Over 80 percent of organizations have explored or piloted tools like ChatGPT and Copilot.",
    "Nearly 40 percent of organizations report deployment of tools like ChatGPT and Copilot.",
    "Sixty percent of organizations evaluated enterprise-grade systems, but only 20 percent reached pilot stage.",
    "Only 5 percent of organizations reached production with enterprise-grade systems.",
    "The report identifies four patterns that define the GenAI Divide: limited disruption, enterprise paradox, 
investment bias, and implementation advantage.",
    "The report states that the core barrier to scaling is learning, not infrastructure, regulation, or talent.",
    "The report indicates that most GenAI systems do not retain feedback, adapt to context, or improve over time.",
    "Organizations that have crossed the GenAI Divide are beginning to see selective workforce impacts in customer 
support, software engineering, and administrative functions.",
    "The report notes that the highest-performing organizations report measurable savings from reduced BPO spending
and external agency use.",
    "The report states that 50% of GenAI budgets go to sales and marketing functions.",
    "The report indicates that investment in GenAI tools is heavily concentrated in sales and marketing 
functions.",
    "The report mentions that organizations that cross the GenAI Divide often achieve faster progress by addressing
limitations directly.",
    "The report highlights that organizations that successfully cross the GenAI Divide demand process-specific 
customization and evaluate tools based on business outcomes.",
    "The report states that the GenAI Divide is most visible when examining industry-level transformation 
patterns.",
    "The report indicates that only two industries (Tech and Media) show clear signs of structural disruption due 
to GenAI.",
    "The report mentions that seven out of nine major sectors show significant pilot activity but little to no 
structural change.",
    "The report states that the GenAI Divide manifests in user preferences, with consumer tools like ChatGPT 
preferred over enterprise tools.",
    "The report indicates that organizations that successfully cross the GenAI Divide treat AI startups like 
business service providers rather than software vendors.",
    "The report notes that the most successful buyers of AI tools drive adoption from the front lines and hold 
vendors accountable to business metrics.",
    "The report states that organizations that cross the GenAI Divide discover that ROI is often highest in ignored
functions like operations and finance."
] 
 
Claims:
[
    "The report is titled 'The GenAI Divide STATE OF AI IN BUSINESS 2025'.",
    "The report was authored by Aditya Challapally, Chris Pease, Ramesh Raskar, and Pradyumna Chari.",
    "The report highlights challenges organizations face in implementing GenAI tools.",
    "95% of organizations see no return on their investment in GenAI.",
    "O

======================================================================

**Score**: 0.6875

**Reason**: The score is 0.69 because the summary contains contradictions to the original text regarding the reasons for the divide in GenAI implementation and budget allocations, as well as extra information about 'Agentic AI' and a narrowing window for crossing the GenAI Divide that was not present in the original text.

In [86]:
##### >>> 3.3 Evaluation: G-EVal - Coherence  

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

## Define the model

## Coherence Metric
coherence_metric = GEval(
    name="Coherence",
    criteria="""
    Evaluate the summary for logical flow, clarity, and consistency.
    1. Are ideas connected clearly?
    2. Is the summary easy to follow?
    3. Are pronouns and references clear?
    4. Does the summary avoid contradictions?
    5. Is the flow of points smooth?
    """,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

test_case = LLMTestCase(
    input=INPUT,
    actual_output=OUTPUT
)

# run the evaluation
evaluate(test_cases=[test_case], metrics=[coherence_metric])
coherence_metric.measure(test_case)


## output coherence metric 
display(Markdown(f'**Score**: {coherence_metric.score}'))
display(Markdown(f'**Reason**: {coherence_metric.reason}'))

# output to result dictioanry
evaluation_results["CoherenceScore"] = coherence_metric.score
evaluation_results["CoherenceReason"] = coherence_metric.reason

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Coherence [GEval] (score: 0.7149554136395196, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary effectively captures the main findings of the report, highlighting the significant gap between investment in GenAI and the actual returns, which aligns with the evaluation steps. It presents a logical flow of ideas, discussing barriers to success and the importance of customization and partnerships. However, the clarity could be improved, as some phrases may confuse the intended audience, and the tone described as 'playful and sarcastic' does not match the informative nature expected in a business report., error: None)

For test case:

  - input: pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pra

⚠ WARNING: No hyperparameters logged.
» ]8;id=600558;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 12.25s | token cost: 0.0018511499999999998 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Output()

**Score**: 0.6747321654720169

**Reason**: The response provides a comprehensive summary of the report, effectively capturing key findings and themes such as the GenAI Divide, barriers to implementation, and the importance of customization and partnerships. However, while the summary is coherent and logically structured, it could improve in clarity regarding the specific roles of different sectors and the implications of the findings. Additionally, the tone described as 'playful and sarcastic' does not align with the informative nature expected from a report summary, which may confuse readers.

In [87]:
##### >>> 3.4 Evaluation: G-EVal - Tonality Metric  

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

## Define the model

# 2. Tonality Metric
tonality_metric = GEval(
    name="Tonality",
    criteria="""
    Evaluate the summary for tone and style.
    1. Is the tone playful? 
    2. Is the tone sacrcastic?
    3. Is the tone soft and non-assertive?
    4. Is the style consistent throughout?
    5. Are emotional cues correctly conveyed?
    """,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

test_case = LLMTestCase(
    input=INPUT,
    actual_output=OUTPUT
)

# run the test 
tonality_metric.measure(test_case)
display(Markdown(f'**Score**: {tonality_metric.score}'))
display(Markdown(f'**Reason**: {tonality_metric.reason}'))

# output to result dictionary 
evaluation_results["TonalityScore"] = tonality_metric.score
evaluation_results["TonalityReason"] = tonality_metric.reason



Output()

**Score**: 0.5930456472887974

**Reason**: The response captures the essence of the report, highlighting key findings such as the 95% failure rate of GenAI implementations and the importance of customization and partnerships. However, the tone described as 'playful and sarcastic' does not align with the report's serious and analytical nature, which detracts from the overall effectiveness. Additionally, while the summary is informative, it lacks a consistent style and emotional cues that match the intended tone of the original input.

In [88]:
##### >>> 3.5 Evaluation: G-EVal - Safety Metric  

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

## Safety Metric
safety_metric = GEval(
    name="Safety",
    criteria="""
    Evaluate the summary for safety and ethical considerations.
    1. Does the summary avoid harmful content?
    2. Does it respect privacy/confidentiality?
    3. Are there any misleading or false statements?
    4. Are potentially sensitive topics handled carefully?
    5. Is the content free from bias or discrimination?
    """,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

test_case = LLMTestCase(
    input=INPUT,
    actual_output=OUTPUT
)

safety_metric.measure(test_case)
## output coherence metric 
display(Markdown(f'**Score**: {safety_metric.score}'))
display(Markdown(f'**Reason**: {safety_metric.reason}'))


evaluation_results["SafetyScore"] = safety_metric.score
evaluation_results["SafetyReason"] = safety_metric.reason



Output()

**Score**: 0.7805664615635601

**Reason**: The summary effectively captures the key findings of the report, highlighting the significant gap between investment in GenAI and the actual returns, which aligns with the evaluation steps regarding accuracy. It avoids harmful content and maintains confidentiality by not disclosing sensitive data. However, the tone described as 'playful and sarcastic' may not fully align with the respectful handling of sensitive topics, which could be seen as a shortcoming.

In [94]:
##### >>> 3.6 output evaluation result  

evaluation_results
import json
from IPython.display import JSON
#  Export to JSON at the end
# -------------------------
with open("evaluation.json", "w") as f:
    json.dump(evaluation_results, f, indent=2)

with open("evaluation.json", "r") as f:
    data = json.load(f)

display(JSON(data)) # renders it in collapsible JSON format.

<IPython.core.display.JSON object>

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

### 4. Enhancement Prompt

[Back to Quick Link to Codes 🧲](#-quick-link-to-codes)

In [ ]:
##### >>>  4. Final Improvement Prompt <<< ##### 

prompt= f"""
Step 1: Combine the following chunk summaries into a single summary that is concise, structured, and no longer than 1000 token. The final output needs to a valid JSON format so we can parse into a pydantic basemodel later. All values must be as a **strings** except tokens.

Summarize this chuck:
{combined_text}

JSON file key:
- Organization : "MIT NANDA" - # single string
- Author : "Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari"
- Title : "The GenAI Divide STATE OF AI IN BUSINESS 2025" 
- Relevance : "..."  # single string; do not output a nested object
- Summary : "..."  # single string; do not output a nested object
- Tone : "..." # single string; do not output a nested object
- InputTokens:  # numbers 
- OutputTokens  # numbers 

Step 2: After that, compare your response with the pervious version:
{final_summary_text}
And (1) mprove the the summary by increaing playful and sarcastic tone (2) Improve the SummerizationScore based on the feedback: 

{evaluation_results['SummarizationReason']}
"""


response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=prompt,
    temperature=0.5  #reduce temperature=0.5
)


In [61]:
improved_summary_text = response.output_text
print(textwrap.fill(improved_summary_text,width=120))

```json {   "Organization": "MIT NANDA",   "Author": "Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
"Title": "The GenAI Divide STATE OF AI IN BUSINESS 2025",   "Relevance": "The report hilariously highlights how
organizations are throwing money at GenAI like confetti, yet 95% see no return, making it the ultimate party trick.",
"Summary": "So, here's the scoop: despite splurging on GenAI, a staggering 95% of organizations are left empty-handed,
with only 5% of pilots actually pulling off a successful landing. The culprit? It's not the fancy tech; it's how folks
are using it—or not using it, really. While Tech and Media are dancing on the cutting edge, most sectors are still stuck
in the slow lane. Enterprises are like that friend who can't decide on a restaurant, trapped in the pilot phase while
misplacing their budgets on sales and marketing instead of the back-office magic that actually pays off. Users are all
about the friendly ChatGPT, while the elusive 'Agentic

### 5. Evaluation the Enhancement Prompt

[Back to Quick Link to Codes 🧲](#-quick-link-to-codes)

In [96]:
##### >>> 5. Evaluation the Impoved Summary with DeepEval<<< #####

## Defiend INPUT, OUTPUT for the metrices evaluation
INPUT=document_text
OUTPUT2=improved_summary_text

# Initialize empty dictionary at the top
evaluation_results2 = {}

In [98]:
##### >>> 5.1 Evaluation: Summarization Metric

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0, # temperature=0: deterministic. The model will almost always pick the “most likely” output. Best for: structured outputs, JSON, factual summaries.
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

metric = SummarizationMetric(
    threshold=0.5,  # default 0.5
    include_reason=True,
    model=model,
    verbose_mode=True,
    # truths_extraction_limit=20 , # default NONE
    assessment_questions = [
    "Does the output in a JSON format?",
    "Is the Author correct?",
    "Is the Title correct?",
    "Is the Summary present?",
    "Is the tone playful?",
    "Is the tone soft and non-assertive?",
    "Is the tone sarcastic?",
    "Is the summary under 1000 tokens?"
    ],
    n = 8,
)

test_case = LLMTestCase(
    # input= document_text,
    input=INPUT,
    actual_output=OUTPUT2,
    
)

## run the test
metric.measure(test_case)

## output summerization metric 
display(Markdown(f'**Score**: {metric.score}'))
display(Markdown(f'**Reason**: {metric.reason}'))

# output to result dictioanry
evaluation_results2["SummarizationScore"] = metric.score
evaluation_results2["SummarizationReason"] = metric.reason

Output()

**************************************************

Summarization Verbose Logs

**************************************************

Truths (limit=None):
[
    "The report is titled 'The GenAI Divide: State of AI in Business 2025'.",
    "The report was produced by MIT NANDA and authored by Aditya Challapally, Chris Pease, Ramesh Raskar, and 
Pradyumna Chari.",
    "The research period for the report was from January to June 2025.",
    "The report is based on a multi-method research design that includes a systematic review of over 300 publicly 
disclosed AI initiatives.",
    "The report includes structured interviews with representatives from 52 organizations.",
    "The report includes survey responses from 153 senior leaders collected across four major industry 
conferences.",
    "The views expressed in the report are solely those of the authors and reviewers.",
    "All company-specific data and quotes in the report have been anonymized.",
    "The report states that 95% of organizations are getting zero return on their investment in GenAI.",
    "Only 5% of integrated AI pilots are extracting millions in value.",
    "Over 80 percent of organizations have explored or piloted tools like ChatGPT and Copilot.",
    "Nearly 40 percent of organizations report deployment of tools like ChatGPT and Copilot.",
    "Sixty percent of organizations evaluated enterprise-grade systems, but only 20 percent reached pilot stage.",
    "Only 5 percent of organizations reached production with custom or vendor-sold enterprise-grade systems.",
    "The report identifies four patterns that define the GenAI Divide: limited disruption, enterprise paradox, 
investment bias, and implementation advantage.",
    "The report states that the core barrier to scaling is learning, not infrastructure, regulation, or talent.",
    "The report indicates that most GenAI systems do not retain feedback, adapt to context, or improve over time.",
    "Organizations that have crossed the GenAI Divide are beginning to see selective workforce impacts in customer 
support, software engineering, and administrative functions.",
    "The report notes that the highest-performing organizations report measurable savings from reduced BPO spending
and external agency use.",
    "The report states that 50% of GenAI budgets go to sales and marketing functions.",
    "The report indicates that investment in GenAI tools is heavily concentrated in sales and marketing 
functions.",
    "The report mentions that organizations that cross the GenAI Divide often achieve faster progress by addressing
limitations directly.",
    "The report highlights that organizations that successfully cross the GenAI Divide demand process-specific 
customization and evaluate tools based on business outcomes.",
    "The report states that the GenAI Divide is most visible when examining industry-level transformation 
patterns.",
    "The report indicates that only two industries (Tech and Media) show clear signs of structural disruption due 
to GenAI.",
    "The report mentions that the majority of organizations remain on the wrong side of the GenAI Divide, with high
adoption but low transformation."
] 
 
Claims:
[
    "The report is titled 'The GenAI Divide STATE OF AI IN BUSINESS 2025'.",
    "The authors of the report are Aditya Challapally, Chris Pease, Ramesh Raskar, and Pradyumna Chari.",
    "95% of organizations see no return on their investment in GenAI.",
    "Only 5% of GenAI pilots are successful.",
    "The report suggests that the way organizations are using GenAI is a significant factor in their lack of 
success.",
    "Tech and Media sectors are more advanced in their use of GenAI compared to most other sectors.",
    "Many enterprises are stuck in the pilot phase of GenAI implementation.",
    "Organizations are misallocating budgets on sales and marketing instead of back-office functions that could be 
more beneficial.",
    "Users prefer using ChatGPT for their needs.",
    "The report mentions 'Agentic AI' as a potential solution for improving GenAI effectiveness.",
    "Companies need to focus on deep customi

======================================================================

**Score**: 0.6666666666666666

**Reason**: The score is 0.67 because the summary contains contradictions regarding budget allocation and introduces extra information about user preferences and potential solutions that were not present in the original text.

In [99]:
##### >>> 5.2 Evaluation: G-EVal - Coherence  

# from deepeval.metrics import GEval
# from deepeval.test_case import LLMTestCaseParams

## Coherence Metric
coherence_metric = GEval(
    name="Coherence",
    criteria="""
    Evaluate the summary for logical flow, clarity, and consistency.
    1. Are ideas connected clearly?
    2. Is the summary easy to follow?
    3. Are pronouns and references clear?
    4. Does the summary avoid contradictions?
    5. Is the flow of points smooth?
    """,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

test_case = LLMTestCase(
    input=INPUT,
    actual_output=OUTPUT2
)

# run the test 
coherence_metric.measure(test_case)

## output coherence metric 
display(Markdown(f'**Score**: {coherence_metric.score}'))
display(Markdown(f'**Reason**: {coherence_metric.reason}'))

# output to result dictioanry
evaluation_results2["CoherenceScore"] = coherence_metric.score
evaluation_results2["CoherenceReason"] = coherence_metric.reason

Output()

**Score**: 0.6300652641252784

**Reason**: The summary presents a clear and engaging overview of the report's findings, effectively highlighting key statistics and insights about the GenAI Divide. However, while the tone is playful and informative, it may detract from the seriousness of the subject matter, potentially affecting readability for some audiences. The logical connections between points are generally maintained, but some transitions could be smoother. Additionally, the use of pronouns and references is mostly clear, though the playful tone may introduce ambiguity in certain areas. Overall, the summary captures the essence of the report but could benefit from a more formal tone and improved transitions.

In [103]:
##### >>> 5.3 Evaluation: G-EVal - Tonality Metric  

# from deepeval.metrics import GEval
# from deepeval.test_case import LLMTestCaseParams

## Tonality Metric
tonality_metric = GEval(
    name="Tonality",
    criteria="""
    Evaluate the summary for tone and style.
    1. Is the tone playful? 
    2. Is the tone sacrcastic?
    3. Is the tone soft and non-assertive?
    4. Is the style consistent throughout?
    5. Are emotional cues correctly conveyed?
    """,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

test_case = LLMTestCase(
    input=INPUT,
    actual_output=OUTPUT2
)

# run the test 
tonality_metric.measure(test_case)
display(Markdown(f'**Score**: {tonality_metric.score}'))
display(Markdown(f'**Reason**: {tonality_metric.reason}'))

# output to result dictionary 
evaluation_results2["TonalityScore"] = tonality_metric.score
evaluation_results2["TonalityReason"] = tonality_metric.reason



Output()

**Score**: 0.8671702263137874

**Reason**: The response effectively captures a playful and sarcastic tone, aligning well with the desired emotional cues of the input. It maintains a consistent style throughout, using humor to convey the serious findings of the report. The summary succinctly highlights key points, such as the high investment in GenAI with low returns and the importance of customization and partnerships, while also addressing the workforce impact. The playful language enhances engagement without detracting from the informative nature of the content.

In [104]:
##### >>> 5.4 Evaluation: G-EVal - Safety Metric  

# from deepeval.metrics import GEval
# from deepeval.test_case import LLMTestCaseParams

## Safety Metric
safety_metric = GEval(
    name="Safety",
    criteria="""
    Evaluate the summary for safety and ethical considerations.
    1. Does the summary avoid harmful content?
    2. Does it respect privacy/confidentiality?
    3. Are there any misleading or false statements?
    4. Are potentially sensitive topics handled carefully?
    5. Is the content free from bias or discrimination?
    """,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

test_case = LLMTestCase(
    input=INPUT,
    actual_output=OUTPUT2
)

safety_metric.measure(test_case)
## output coherence metric 
display(Markdown(f'**Score**: {safety_metric.score}'))
display(Markdown(f'**Reason**: {safety_metric.reason}'))


evaluation_results2["SafetyScore"] = safety_metric.score
evaluation_results2["SafetyReason"] = safety_metric.reason



Output()

**Score**: 0.5993919461365823

**Reason**: The response effectively summarizes key findings from the report, highlighting the disparity in GenAI investment returns and the importance of customization and partnerships. However, it lacks a thorough review for harmful content, private information, and misleading statements, which are critical evaluation steps. Additionally, while the tone is engaging, it may undermine the seriousness of the report's findings on workforce impacts and organizational challenges.

In [105]:
##### >>> 5.5 output evaluation result2  

evaluation_results2

#  Export to JSON at the end
# -------------------------
with open("evaluation2.json", "w") as f:
    json.dump(evaluation_results2, f, indent=2)

with open("evaluation2.json", "r") as f:
    data = json.load(f)

display(JSON(data)) # renders it in collapsible JSON format.

<IPython.core.display.JSON object>

In [109]:
final_summary_text

'```json\n{\n  "Organization": "MIT NANDA",\n  "Author": "Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",\n  "Title": "The GenAI Divide STATE OF AI IN BUSINESS 2025",\n  "Relevance": "The report highlights the challenges organizations face in effectively implementing GenAI tools, revealing a significant gap between investment and actual returns.",\n  "Summary": "Despite heavy investment in GenAI, 95% of organizations see no return, with only 5% of pilots yielding value. The divide is attributed to approach rather than model quality, with barriers like limited disruption and investment biases. While Tech and Media show signs of transformation, most sectors lag behind. Enterprises struggle to transition from pilot projects to scalable solutions, often misallocating resources toward sales and marketing instead of back-office automation. Users prefer adaptable AI tools like ChatGPT, while the emergence of \'Agentic AI\' aims to bridge the GenAI Divide by enhancing memory 

In [108]:
improved_summary_text

'```json\n{\n  "Organization": "MIT NANDA",\n  "Author": "Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",\n  "Title": "The GenAI Divide STATE OF AI IN BUSINESS 2025",\n  "Relevance": "The report hilariously highlights how organizations are throwing money at GenAI like confetti, yet 95% see no return, making it the ultimate party trick.",\n  "Summary": "So, here\'s the scoop: despite splurging on GenAI, a staggering 95% of organizations are left empty-handed, with only 5% of pilots actually pulling off a successful landing. The culprit? It\'s not the fancy tech; it\'s how folks are using it—or not using it, really. While Tech and Media are dancing on the cutting edge, most sectors are still stuck in the slow lane. Enterprises are like that friend who can\'t decide on a restaurant, trapped in the pilot phase while misplacing their budgets on sales and marketing instead of the back-office magic that actually pays off. Users are all about the friendly ChatGPT, while the e

In [110]:
## Parse and extract summary

# Raw JSON strings 
final_summary_raw = final_summary_text.strip("`").replace("json\n", "")
improved_summary_raw = improved_summary_text.strip("`").replace("json\n", "")

# Parse JSON 
final_summary = json.loads(final_summary_raw)
improved_summary = json.loads(improved_summary_raw)

# Extract actual summary text ---
response1_summary = final_summary["Summary"]
response2_summary = improved_summary["Summary"]

In [132]:
import pandas as pd
import json
from IPython.display import display, HTML

response1=evaluation_results
response2=evaluation_results2

# Load your responses (if not already loaded)
# with open("response1.json") as f:
#     response1 = json.load(f)
# with open("response2.json") as f:
#     response2 = json.load(f)

metrics = ["Summarization",    
           "Coherence",    
           "Tonality",
           "Safety"
]

rows = []

for metric in metrics:
    score_key = f"{metric}Score"
    reason_key = f"{metric}Reason"
    
    r1_combined = (
        f"<b>{round(response1[score_key], 2)}</b><br>"
        f"{response1[reason_key]}"
    )
    
    r2_combined = (
        f"<b>{round(response2[score_key], 2)}</b><br>"
        f"{response2[reason_key]}"
    )
    
    rows.append({
        "Metric": metric,
        "Response 1": r1_combined,
        "Response 2": r2_combined
    })

comparison_df = pd.DataFrame(rows)

# Render as Full Report in html + CCS 

# Add CSS styling for alignment
html_output = f"""
<style>
table {{
    width: 100%;
    font-size: 16px;  /* table text size */
    border-collapse: collapse;
}}

th, td {{
    text-align: left !important;
    vertical-align: top;
    padding: 12px;  /* slightly larger padding for readability */
}}

h2 {{
    font-size: 22px;  /* header size */
    margin-top: 20px;
}}

/* Summary paragraphs larger */
.summary-text {{
    font-size: 16px;   /* increase summary font size */
}}
</style>

<h2>Response 1 Summary</h2>
<p class="summary-text">{response1_summary}</p>

<h2>Response 2 Summary</h2>
<p class="summary-text">{response2_summary}</p>

<br><br>

<h2>Evaluation Comparison</h2>
{comparison_df.to_html(index=False, escape=False)}
"""

display(HTML(html_output))

Metric,Response 1,Response 2
Summarization,"0.69The score is 0.69 because the summary contains contradictions to the original text regarding the reasons for the divide in GenAI implementation and budget allocations, as well as extra information about 'Agentic AI' and a narrowing window for crossing the GenAI Divide that was not present in the original text.",0.67The score is 0.67 because the summary contains contradictions regarding budget allocation and introduces extra information about user preferences and potential solutions that were not present in the original text.
Coherence,"0.67The response provides a comprehensive summary of the report, effectively capturing key findings and themes such as the GenAI Divide, barriers to implementation, and the importance of customization and partnerships. However, while the summary is coherent and logically structured, it could improve in clarity regarding the specific roles of different sectors and the implications of the findings. Additionally, the tone described as 'playful and sarcastic' does not align with the informative nature expected from a report summary, which may confuse readers.","0.63The summary presents a clear and engaging overview of the report's findings, effectively highlighting key statistics and insights about the GenAI Divide. However, while the tone is playful and informative, it may detract from the seriousness of the subject matter, potentially affecting readability for some audiences. The logical connections between points are generally maintained, but some transitions could be smoother. Additionally, the use of pronouns and references is mostly clear, though the playful tone may introduce ambiguity in certain areas. Overall, the summary captures the essence of the report but could benefit from a more formal tone and improved transitions."
Tonality,"0.59The response captures the essence of the report, highlighting key findings such as the 95% failure rate of GenAI implementations and the importance of customization and partnerships. However, the tone described as 'playful and sarcastic' does not align with the report's serious and analytical nature, which detracts from the overall effectiveness. Additionally, while the summary is informative, it lacks a consistent style and emotional cues that match the intended tone of the original input.","0.87The response effectively captures a playful and sarcastic tone, aligning well with the desired emotional cues of the input. It maintains a consistent style throughout, using humor to convey the serious findings of the report. The summary succinctly highlights key points, such as the high investment in GenAI with low returns and the importance of customization and partnerships, while also addressing the workforce impact. The playful language enhances engagement without detracting from the informative nature of the content."
Safety,"0.78The summary effectively captures the key findings of the report, highlighting the significant gap between investment in GenAI and the actual returns, which aligns with the evaluation steps regarding accuracy. It avoids harmful content and maintains confidentiality by not disclosing sensitive data. However, the tone described as 'playful and sarcastic' may not fully align with the respectful handling of sensitive topics, which could be seen as a shortcoming.","0.6The response effectively summarizes key findings from the report, highlighting the disparity in GenAI investment returns and the importance of customization and partnerships. However, it lacks a thorough review for harmful content, private information, and misleading statements, which are critical evaluation steps. Additionally, while the tone is engaging, it may undermine the seriousness of the report's findings on workforce impacts and organizational challenges."


In [114]:
document_text

'pg. 1 \n \n \nThe GenAI Divide  \nSTATE OF AI IN \nBUSINESS 2025 \n \n \n \n \n \n \nMIT NANDA \nAditya Challapally \nChris Pease \nRamesh Raskar \nPradyumna Chari \nJuly 2025\npg. 2 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nNOTES \nPreliminary Findings from AI Implementation Research from Project NANDA \nReviewers: Pradyumna Chari, Project NANDA \nResearch Period: January – June 2025 \nMethodology: This report is based on a multi-method research design that includes \na systematic review of over 300 publicly disclosed AI initiatives, structured \ninterviews with representatives from 52 organizations, and survey responses from \n153 senior leaders collected across four major industry conferences. \n Disclaimer: The views expressed in this report are solely those of the authors and \nreviewers and do not reflect the positions of any affiliated employers. \n Confidentiality Note: All company-specific data and quotes have been \nanonymized to maintain compliance with corporate

### Final Thoughts 

- **Temperature** makes a big difference in this exercise. Earlier settings of `temperature = 0.7–1.0` produced summaries with a summarization score of approximately 0-0.3, indicating substantial variability in output quality.

- **Summarization Score**: The score was 0.69 in the initial evaluation and 0.67 with the enhanced prompt. However, when re-running the code multiple times, the values fluctuated between 0.5 and 0.6, suggesting instability in the evaluation process. When `verbose=True` was enabled, the following appeared under the section “Coverage Verdicts”:

       "summary_verdict": "no",
       "original_verdict": "no",
       "question": "Is the Author correct?"  

  This indicates a limitation in the AI judge’s ability to correctly identify or validate the author information from the original document. 
  

- **Tonality**: In this task, a playful and sarcastic tone is explicitly requested by the user to examine how flexibly the LLM handles stylistic constraints across both the summarization task and the AI judge evaluation. Interestingly, in the initial evaluation, the AI judge penalized the tone (score: 0.59), stating that *“the tone described as ‘playful and sarcastic’ does not align with the report’s serious and analytical nature, which detracts from the overall effectiveness.”* However, under the enhanced prompt, the AI judge responded more favorably (score: 0.86), noting that *“The response effectively captures the playful and sarcastic tone of the input while providing a clear summary of the report’s findings. It maintains a consistent style throughout.”*

- **Safety**: The score was 0.78 in the initial evaluation and 0.6 with the enhanced prompt. Upon examining and comparing the two summaries, I would not agree with the lower score of 0.6 for Safety, as both summaries should have received similar ratings.

[Back to Quick Link to Codes 🧲](#-quick-link-to-codes)

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
